In [1]:
import os
import random
import torch
import pybiber as pb
import polars as pl
import pandas as pd
import numpy as np
from transformers import pipeline
from scipy.stats import zscore, pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, matthews_corrcoef
from sklearn.model_selection import train_test_split

# HuggingFace Login
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)
# Set up transformers logging (set it to minimal logging).
from transformers import logging
logging.set_verbosity_error()

In [2]:
# Free memory
torch.cuda.empty_cache()

In [3]:
# Constant Variables.
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR_TRAIN = 'outputsTrain'
OUTPUT_DIR_TEST = 'outputsTest'
OUTPUT_DIR_ALL = 'outputsAll'
# 100 samples (total) chosen for comparability purposes with "Measuring the Measuring Tool" by Kour et al. (https://doi.org/10.18653/v1/2022.gem-1.35). We will do 10 x 100. This also allows for the test and validation sets to be adequately compared.
SAMPLE_SIZE = 10 # 10 * 10 = 100 
BATCH_SIZE = 8

ZERO_SHOT_MODELS = [
    "cross-encoder/nli-deberta-v3-small", # low capacity
    "typeform/distilbert-base-uncased-mnli", # medium capacity
    "valhalla/distilbart-mnli-12-3", # higher capacity
]

CLASSIFIERS = {
    model_name: pipeline(
    "zero-shot-classification",
    model=model_name,
    device=DEVICE
)
for model_name in ZERO_SHOT_MODELS
}

# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {
        "informational, dense, precise": -1,
        "involved, interactive, affective": 1
    },
    "factor_2": {
        "non-narrative, expository, informational": -1,
        "narrative, event-focused, storytelling": 1
    },
    "factor_3": {
        "situation-dependent, context-bound, implicit": -1,
        "explicit, context-independent, elaborated": 1
    },
    "factor_4": {
        "non-persuasive, non-argumentative, neutral": -1,
        "persuasive, argumentative, modalized": 1
    },
    "factor_5": {
        "non-abstract, concrete, human-centered": -1,
        "abstract, impersonal, technical": 1
    },
    "factor_6": {
        "compressed, dense, clause-poor": -1,
        "elaborated, expanded, clause-rich": 1
    }
}

# Flatten Biber label map (faster inference).
all_labels = []
label_to_factor = {}

for factor, description in BIBER_LABEL_MAP.items():
    for label in description.keys():
        all_labels.append(label)
        label_to_factor[label] = factor


# Using different prompt templates increases robustness.
TEMPLATES = ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [5]:
# Load all data.
list_of_dfs_train = []
list_of_dfs_test = []
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    folder_path = f'./{FILE_PATH}/{folder}'
    if not os.path.isdir(folder_path):
        continue

    for file in os.listdir(folder_path):
        if not file.endswith(".csv"):
            continue

        file_path = f"{folder_path}/{file}"
        temp_df = pl.read_csv(file_path)

        # Ensure text column exists.
        if 'text' not in temp_df.columns:
            continue

        # Convert all values to string and remove NaNs.
        temp_texts = [
            str(t).strip() 
            for t in temp_df['text'].to_list() 
            if t is not None and str(t).strip() != '' and str(t).strip().lower() != 'nan'
            and len(str(t).strip()) >= 50   # Make sure texts are longer than 50 characters (to reduce the likelihood that Biber drops texts which are too short).
        ]

        # Skip if no valid text.
        if not temp_texts:
            print(f"There are no valid texts for dataset {file}.")
            continue

        # Shuffle and perform 80-20 train-test split.
        temp_train, temp_test = train_test_split(
            temp_texts, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
        )

        # Make unique document ids.
        temp_tag = file.replace('.csv', '')

        def build_polars_df(texts):
            df = pl.DataFrame({'text': texts})
            df = df.with_row_index("index_num")  # Add row index.
            df = df.with_columns(
                (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
            )
            df = df.with_columns(
                pl.col("doc_id").str.split("_").list.get(0).alias("tag")
            )
            return df.select(['text', 'doc_id', 'tag'])

        list_of_dfs_train.append(build_polars_df(temp_train))
        list_of_dfs_test.append(build_polars_df(temp_test))
        list_of_dfs.append(build_polars_df(temp_texts))

# Combine all train-test dfs (separately).
combined_train = pl.concat(list_of_dfs_train, how="vertical")
combined_test = pl.concat(list_of_dfs_test, how="vertical")
combined_all = pl.concat(list_of_dfs, how="vertical")

In [ ]:
def make_data(combined, output_dir):
    combined = combined.to_pandas()
    for random_change in range(100):
        # try:   
        # Randomly sample from dataframe.
        temp_df = (combined.groupby("tag")).apply(lambda x: x.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE + random_change))
        temp_df = pl.from_pandas(temp_df)

        # Set up df for use.
        df = temp_df.select(["doc_id", "text"])

        # Light preprocessing to strip extra whitespace.
        df = df.with_columns(
            pl.col("text")
            .str.strip_chars()
            .str.replace_all(r"\s+", " ")
            .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
            .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
        )
        # Make pybiber pipeline.
        pybiber_pipeline = pb.PybiberPipeline(model="en_core_web_sm")
        features, _ = pybiber_pipeline.run(df, return_tokens=True)
        # Full feature list can be found here: https://browndw.github.io/pybiber/feature-categories.html
        features = features.with_columns(pl.col("doc_id").str.split("_").list.get(0).alias("category"))
        

        # Statistical analysis and visualization
        analyzer = pb.BiberAnalyzer(features, id_column='category')

        # Multi-Dimensional Analysis - see https://browndw.github.io/pybiber/biber-analyzer.html#comparison-with-bibers-original-dimensions for factor mapping
        # Explanation of the factor mapping to dimensions can be found here: https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html
        '''
        Factor 1: Involved vs. Informational Production (negative to positive)
        Factor 2: Narrative vs. Non-narrative Concerns (negative to positive)
        Factor 3: Explicit vs. Situation-dependent Reference (negative to positive)
        Factor 4: Overt Expression of Persuasion (negative to positive)
        Factor 5: Abstract vs. Non-abstract Information (negative to positive)
        Factor 6: On-line Informational Elaboration (negative to positive)
        '''

        analyzer.mda_biber()

        # Get Z-Scores from Biber analysis.
        biber_dimensions = (analyzer.mda_dim_scores).to_pandas()
        # Get factor columns.
        factor_cols = [c for c in biber_dimensions.columns if c.startswith("factor")]
        biber_dimensions[factor_cols] = biber_dimensions[factor_cols].apply(zscore)
        for c in factor_cols:
            biber_dimensions[f"{c}_label"] = biber_dimensions[c] > 0
        biber_dimensions = pl.from_pandas(biber_dimensions)

        temp_df = temp_df.to_pandas()
        texts = temp_df['text'].values.tolist()
        doc_ids = temp_df['doc_id'].values.tolist()

        assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."

        all_dfs = []

        for model_name, classifier in CLASSIFIERS.items():
            # Initialize factor storage
            temp_factors_list = {
                'model_name': [model_name] * len(doc_ids),
                'doc_id': doc_ids,
                **{factor: [] for factor in BIBER_LABEL_MAP}
            }

            # Dictionary to accumulate scores per factor across templates
            factor_scores_accum = {factor: [0.0] * len(texts) for factor in BIBER_LABEL_MAP}
            # Loop over all templates
            for template in TEMPLATES:

                with torch.no_grad():
                    outputs = classifier(
                        texts,
                        candidate_labels=all_labels,
                        hypothesis_template=template,
                        multi_label=True,
                        batch_size = BATCH_SIZE
                    )

                    if isinstance(outputs, dict):
                        outputs = [outputs]

                    # Accumulate weighted scores per factor
                    for j, output in enumerate(outputs):
                        for label, score in zip(output['labels'], output['scores']):
                            f = label_to_factor[label]
                            weight = BIBER_LABEL_MAP[f][label]
                            factor_scores_accum[f][j] += weight * score

            # Average over templates
            n_templates = len(TEMPLATES)
            for factor in BIBER_LABEL_MAP:
                factor_scores_accum[factor] = [s / n_templates for s in factor_scores_accum[factor]]
                temp_factors_list[factor].extend(factor_scores_accum[factor])

            # Convert to DataFrame and append to all_dfs
            all_dfs.append(pd.DataFrame(temp_factors_list))

        df = pd.concat(all_dfs)

        # Simple averaging will prevent over-confidence. 
        mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()

        # # Get factor columns.
        factor_cols = [c for c in mean_scores.columns if c.startswith("factor")]
        # # Get Z-Scores from zero-shot analysis.
        mean_scores[factor_cols] = mean_scores[factor_cols].apply(zscore)
        for c in factor_cols:
            mean_scores[f"{c}_label"] = mean_scores[c] > 0
        mean_scores = pl.from_pandas(mean_scores)
        results_cont = {}
        for f in ["factor_1", "factor_2", "factor_3", "factor_4", "factor_5", "factor_6"]:
            pearson = pearsonr(biber_dimensions[f], mean_scores[f])[0]
            spearman = spearmanr(biber_dimensions[f], mean_scores[f])[0]
            mse = mean_squared_error(biber_dimensions[f], mean_scores[f])
            rmse = root_mean_squared_error(biber_dimensions[f], mean_scores[f])
            mae = mean_absolute_error(biber_dimensions[f], mean_scores[f])
            results_cont[f] = {"pearson": pearson, "spearman": spearman, "MSE": mse, "RMSE": rmse, "MAE": mae}
        continuous_df = pd.DataFrame(results_cont).T
        continuous_df = continuous_df.rename(index={
            "factor_1": "informational_vs_involved",
            "factor_2": "non-narrative_vs_narrative",
            "factor_3": "situation-dependent_vs_explicit",
            "factor_4": "non-persuasive_vs_persuasive",
            "factor_5": "non-abstract_vs_abstract",
            "factor_6": "compressed_vs_elaborated"
        })
        continuous_df['dimension'] = continuous_df.index
        continuous_df = pl.from_pandas(continuous_df)

        results_bin = {}
        for f in ["factor_1_label", "factor_2_label", "factor_3_label", "factor_4_label", "factor_5_label", "factor_6_label"]:
            acc = accuracy_score(biber_dimensions[f], mean_scores[f])
            prec = precision_score(biber_dimensions[f], mean_scores[f])
            rec = recall_score(biber_dimensions[f], mean_scores[f])
            f1 = f1_score(biber_dimensions[f], mean_scores[f])
            kappa = cohen_kappa_score(biber_dimensions[f], mean_scores[f])
            mcc = matthews_corrcoef(biber_dimensions[f], mean_scores[f])
            results_bin[f] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "kappa": kappa, "MCC": mcc}

        classification_df = pd.DataFrame(results_bin).T
        classification_df = classification_df.rename(index={
            "factor_1_label": "informational_vs_involved",
            "factor_2_label": "non-narrative_vs_narrative",
            "factor_3_label": "situation-dependent_vs_explicit",
            "factor_4_label": "non-persuasive_vs_persuasive",
            "factor_5_label": "non-abstract_vs_abstract",
            "factor_6_label": "compressed_vs_elaborated"
        })
        classification_df['dimension'] = classification_df.index
        classification_df = pl.from_pandas(classification_df)

        # Make directory for saving. 
        output_file_path = f"{output_dir}/{RANDOM_STATE + random_change}"
        os.makedirs(f"./{output_file_path}/", exist_ok=True)

        # Save texts used and their ids.
        temp_df = pl.from_pandas(temp_df)
        # Add category column. 
        temp_df = temp_df.with_columns(
            pl.col("doc_id").str.extract(r"(^[^_]+)").alias("category")
        )
        temp_df.write_csv(f"./{output_file_path}/texts_and_ids.csv", float_precision=15)
        
        # Save biber results.
        biber_dimensions.write_csv(f"./{output_file_path}/mda_dim_scores.csv", float_precision=15)
        analyzer.mda_loadings.write_csv(f"./{output_file_path}/mda_loadings.csv", float_precision=15)
        
        # Save zero-shot results.
        df = pl.from_pandas(df)
        df.write_csv(f"./{output_file_path}/all_model_zero_shot_classification.csv", float_precision=15)
        
        mean_scores.write_csv(f"./{output_file_path}/mean_model_zero_shot_classification.csv", float_precision=15)
        # Save comparison results.
        continuous_df.write_csv(f"./{output_file_path}/continuous_comparison_results_zero_vs_biber.csv", float_precision=15)
        
        classification_df.write_csv(f"./{output_file_path}/classification_comparison_results_zero_vs_biber.csv", float_precision=15)
        
        print(f"Done: {random_change + 1} of 100")
        # except Exception as e:
        #     print(f"There was an error.\nError message: {e}")

make_data(combined_train, OUTPUT_DIR_TRAIN)
make_data(combined_test, OUTPUT_DIR_TEST)
make_data(combined_all, OUTPUT_DIR_ALL)

[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_22_that_adj_comp', 'f_32_wh_obj', 'f_34_sentence_relatives']


Done: 1 of 100


[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_61_stranded_preposition']


Done: 2 of 100


[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_32_wh_obj']


...

[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_22_that_adj_comp', 'f_26_past_participle']


Done: 56 of 100


[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_34_sentence_relatives', 'f_61_stranded_preposition']


Done: 57 of 100


[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_22_that_adj_comp', 'f_26_past_participle', 'f_32_wh_obj']
